In [ ]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import pickle
from tqdm import tqdm

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [46]:
mg_mapping = pd.read_csv('../../SATURN_mapping/mouse_zebrafish_30_seeds.csv', index_col = 'barcode')

In [47]:
mo_mapping = pd.read_csv('../../SATURN_mapping/vole_zebrafish_30_seeds.csv', index_col = 'barcode')

In [48]:
sam = SAM()
sam.load_data('../../Active_SAM_joined/SAM_DR_ncbi_joined_cleaned_07172026.h5ad')

In [49]:
mg_mapping.columns

Index(['seed_0', 'seed_1', 'seed_2', 'seed_3', 'seed_4', 'seed_5', 'seed_6',
       'seed_7', 'seed_8', 'seed_9', 'seed_10', 'seed_11', 'seed_12',
       'seed_13', 'seed_14', 'seed_15', 'seed_16', 'seed_17', 'seed_18',
       'seed_19', 'seed_20', 'seed_21', 'seed_22', 'seed_23', 'seed_24',
       'seed_25', 'seed_26', 'seed_27', 'seed_28', 'seed_29'],
      dtype='object')

In [50]:
sam.adata.obs.columns

Index(['n_genes', 'n_counts', 'key', 'leiden_clusters_neuron', 'eq_subclass',
       'eq_subclass_lc', 'leiden_clusters', 'eq_subclass_nounlabeled',
       'ss_subclass', 'ss_subclass_v4_nounlabeled',
       'ss_subclass_v4_nounlabeled_nn', 'ss_subclass_nounlabeled_nmm_v4_nn',
       'ss_subclass_nounlabeled_nmm_v4_nn_thresh30',
       'ss_subclass_nounlabeled_nmm_cl_v4_nn'],
      dtype='object')

In [51]:
level = 'eq_subclass_lc'

In [52]:
cj_clusters = sam.adata.obs[level].to_frame()
cj_clusters.colums = [level]

/scratch/miniconda/lib/python3.7/site-packages/ipykernel_launcher.py:2: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  


In [53]:
cj_clusters

,eq_subclass_lc
AAACCTGAGAGTTGGC,83
AAACCTGAGGTGACCA,2
AAACCTGCAAGCGTAG,200
AAACCTGCAGTAGAGC,2
AAACCTGCATCACCCT,78
...,...
TTTGTCACACACCGCA,440
TTTGTCACATGTAAGA,260
TTTGTCACATTATCTC,21
TTTGTCAGTCCGAGTC-2,134


In [54]:
cj_clusters[level].unique()

array([ 83,   2, 200,  78,  51, 201,  66,   5, 377, 244,   0, 143, 231,
       195, 176,  23,  84, 245, 172, 339,   1,  16, 210,   3, 312, 300,
       324, 155,   7,  14,  30, 334, 428, 404, 354, 445, 118, 271,  85,
       285, 166,   4, 156,  79,  31, 212, 403, 127, 541, 549,  20,  99,
        67, 219,  64, 158,  65, 500,  76, 121, 242,  55, 240, 153, 134,
       492, 521, 292,  57, 407, 355, 417,  63, 392, 185,  42, 180,  80,
       113, 535,  17,  89, 170,  92, 123,  41,  59, 114,  96, 133, 145,
       580, 547,  40,  39,  60, 457,  38,  46, 254, 189, 452, 218,  34,
       365, 237, 523, 162,   8, 164, 328, 227, 191,  12, 382,  71,  10,
        54, 115, 119, 305, 262, 440,  24, 522, 337,  32, 597, 140, 284,
       349, 199,  44, 304, 565, 178, 150, 514, 494, 414, 224,  52, 232,
       247,  43, 206, 487, 137, 259, 517, 205, 136,  50, 104, 281,  72,
       214,  56, 431, 350, 131, 217, 432, 110, 173, 340, 427, 485, 192,
       160,  73, 209, 443,  35, 348, 374, 148,  33, 258, 279, 35

In [55]:
df_comb_mapping = pd.DataFrame(index = cj_clusters.index, columns = ['seed_' + str(i) for i in range(30)])

In [56]:
cj_clusters

,eq_subclass_lc
AAACCTGAGAGTTGGC,83
AAACCTGAGGTGACCA,2
AAACCTGCAAGCGTAG,200
AAACCTGCAGTAGAGC,2
AAACCTGCATCACCCT,78
...,...
TTTGTCACACACCGCA,440
TTTGTCACATGTAAGA,260
TTTGTCACATTATCTC,21
TTTGTCAGTCCGAGTC-2,134


In [58]:
for i in tqdm(range(30)):
    mapping_dict = {}
    for lc in range(cj_clusters[level].nunique()):
        mg_mapping_set = mg_mapping.loc[cj_clusters[cj_clusters[level] ==lc].index,'seed_' + str(i)]
        mo_mapping_set = mo_mapping.loc[cj_clusters[cj_clusters[level] ==lc].index,'seed_' + str(i)]
        if mg_mapping_set.mode()[0] == mo_mapping_set.mode()[0] and len(mg_mapping_set) >25:
            mapping_dict[lc] = mg_mapping_set.mode()[0] 
        else:
            mapping_dict[lc] = 'Unlabeled'
    new_mapping = [mapping_dict[item] for item in cj_clusters[level]]
    df_comb_mapping.loc[:,'seed_' + str(i)] = new_mapping

100%|███████████████████████████████████████████| 30/30 [00:28<00:00,  1.06it/s]


In [82]:
df_comb_mapping.to_csv('../../SATURN_mapping/MV_mm_mo_SATURN_30_seed.csv')